# Grounding AI With Your Own Data: RAG Project

This notebook builds a complete **RAG (Retrieval-Augmented Generation)** pipeline step by step.

**Business used (dummy data):** *BrightCart*, a fictional online store. Its handbook (`brightcart_handbook.pdf`) has shipping, returns, warranty, support and pricing information. All data is made up.

**Why RAG?** An LLM like ChatGPT knows nothing about BrightCart's private policies. RAG fixes this: we find the relevant pieces of our own data and give them to the LLM, so its answer is *grounded* in our data.

**Pipeline:**
1. Load the PDF
2. Split it into chunks
3. Create embeddings (numbers that capture meaning)
4. Store them in a vector database (Qdrant) - this is *indexing*
5. Retrieve the chunks most similar to a question
6. Send question + chunks to the LLM to get the answer

**Tools:** LangChain, OpenAI (embeddings + ChatGPT), Qdrant.

## Step 1: Setup
Import the libraries and load the secret keys from the `.env` file.
The `.env` file is listed in `.gitignore`, so our keys never go to GitHub.

In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Reads OPENAI_API_KEY, QDRANT_URL and QDRANT_API_KEY from the .env file
load_dotenv()

PDF_PATH = "brightcart_handbook.pdf"   # our dummy company data
COLLECTION = "brightcart_rag"          # name of the collection inside Qdrant

C:\Users\Ahmed\AppData\Local\Temp\ipykernel_20696\157531733.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Step 2: Load the data
`PyPDFLoader` reads the PDF and returns one `Document` per page.
Each document has the page text (`page_content`) and details like the page number (`metadata`).

In [2]:
docs = PyPDFLoader(PDF_PATH).load()

print("Total pages:", len(docs))
print("\nFirst 300 characters of page 1:\n")
print(docs[0].page_content[:300])

Total pages: 1

First 300 characters of page 1:

BrightCart - Company Handbook (Dummy Data)
BrightCart is a fictional online store founded in 2019 in Lahore, Pakistan. We sell home appliances, kitchen
items and smart gadgets. We have 120 employees and serve customers in Pakistan, UAE and Saudi Arabia.
Our mission is to make quality products afford


## Step 3: Split the text into chunks
A whole page is too big to search precisely. `RecursiveCharacterTextSplitter` cuts the text into smaller chunks, trying to split at paragraphs and sentences first.

- `chunk_size=500`: maximum characters per chunk
- `chunk_overlap=100`: neighbouring chunks share 100 characters so an idea is not cut in half

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)

print("Total chunks:", len(chunks))
print("\nExample chunk:\n")
print(chunks[1].page_content)

Total chunks: 6

Example chunk:

take 7 to 10 working days. Shipping is free for orders above Rs 5,000 in Pakistan. For smaller orders the
shipping fee is Rs 250. Customers receive a tracking link by SMS and email once the order is dispatched.
2. Return and Refund Policy
Customers can return a product within 14 days of delivery if it is unused and in its original packaging.
Refunds are processed within 5 working days after we receive the returned item. Refunds go back to the


## Step 4: Create the embedding model
An embedding turns text into a list of numbers (a vector). Texts with similar *meaning* get similar vectors, so we can search by meaning, not just by exact words.
We use OpenAI's `text-embedding-3-small`.

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# See what an embedding looks like
vector = embeddings.embed_query("What is the return policy?")
print("Vector length:", len(vector))
print("First 5 numbers:", vector[:5])

Vector length: 1536
First 5 numbers: [0.01329803466796875, 0.035247802734375, -0.012420654296875, 0.0262451171875, -0.028076171875]


## Step 5: Index the chunks in Qdrant (vector database)
Here every chunk is converted to a vector and stored in Qdrant.
`force_recreate=True` deletes the old collection first, so running this cell again does not create duplicate chunks.

In [5]:
vector_store = QdrantVectorStore.from_documents(
    chunks,
    embeddings,
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    collection_name=COLLECTION,
    force_recreate=True,
)
print("Indexed", len(chunks), "chunks in Qdrant")

Indexed 6 chunks in Qdrant


## Step 6: Retrieval
A *retriever* takes a question, converts it to a vector, and returns the `k` most similar chunks from Qdrant.
Let us test it before adding the LLM.

In [6]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

question = "How long do I have to return a product?"
results = retriever.invoke(question)

for i, doc in enumerate(results, 1):
    print(f"--- Retrieved chunk {i} ---")
    print(doc.page_content, "\n")

--- Retrieved chunk 1 ---
original payment method. Products marked 'Final Sale' and opened personal care items cannot be returned.
3. Warranty
All electronic items come with a 12-month warranty. Kitchen appliances such as blenders and air fryers have
a 24-month warranty. The warranty covers manufacturing defects only. Damage caused by water, drops or
misuse is not covered. To claim a warranty, contact support with your order number.
4. Customer Support 

--- Retrieved chunk 2 ---
take 7 to 10 working days. Shipping is free for orders above Rs 5,000 in Pakistan. For smaller orders the
shipping fee is Rs 250. Customers receive a tracking link by SMS and email once the order is dispatched.
2. Return and Refund Policy
Customers can return a product within 14 days of delivery if it is unused and in its original packaging.
Refunds are processed within 5 working days after we receive the returned item. Refunds go back to the 

--- Retrieved chunk 3 ---
4. Customer Support
Our support team is 

## Step 7: Connect the LLM (generation)
We write a prompt that tells ChatGPT to answer **only from the retrieved context**. This keeps the answer grounded and avoids made-up facts.

In [7]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below. "
    "If the answer is not in the context, say 'I don't know'.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def format_docs(docs):
    # Join the retrieved chunks into one block of text
    return "\n\n".join(d.page_content for d in docs)

## Step 8: Build the RAG chain
The chain runs in this order: **question -> retriever -> prompt -> LLM -> text answer**.

In [8]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## Step 9: Ask questions
The answers come from the BrightCart handbook. The last question is *not* in the handbook, to show that the model says "I don't know" instead of guessing.

In [9]:
questions = [
    "How long do I have to return a product?",
    "How much is shipping for an order of Rs 3,000 in Pakistan?",
    "What is the warranty on an air fryer?",
    "Which payment methods do you accept?",
    "What is the capital of France?",   # not in our data
]

for q in questions:
    print("Q:", q)
    print("A:", rag_chain.invoke(q), "\n")

Q: How long do I have to return a product?


A: Customers can return a product within 14 days of delivery if it is unused and in its original packaging. 

Q: How much is shipping for an order of Rs 3,000 in Pakistan?


A: The shipping fee for an order of Rs 3,000 in Pakistan is Rs 250. 

Q: What is the warranty on an air fryer?


A: The warranty on an air fryer is 24 months. 

Q: Which payment methods do you accept?


A: We accept credit and debit cards, JazzCash, Easypaisa, and bank transfer. Cash on delivery is available in Pakistan for orders up to Rs 50,000. 

Q: What is the capital of France?


A: I don't know. 



## Summary
We loaded a PDF, split it into chunks, created embeddings, indexed them in Qdrant, retrieved relevant chunks for a question, and let ChatGPT answer using only that context. This is how we ground an AI with our own data.